In [0]:
df = spark.read.csv("/Volumes/insurance/dbo/insurance/medical_insurance.csv", header=True, inferSchema=True)

In [0]:
display(df.limit(10))

age,gender,bmi,children,discount_eligibility,region,expenses,premium
19,female,27.9,0,yes,southwest,16884.92,168.8492
18,male,33.8,1,no,southeast,1725.55,17.2555
28,male,33.0,3,no,southeast,4449.46,44.4946
33,male,22.7,0,no,northwest,21984.47,439.6894
32,male,28.9,0,no,northwest,3866.86,77.3372
31,female,25.7,0,no,southeast,3756.62,75.1324
46,female,33.4,1,no,southeast,8240.59,164.8118
37,female,27.7,3,no,northwest,7281.51,145.6302
37,male,29.8,2,no,northeast,6406.41,128.1282
60,female,25.8,0,no,northwest,28923.14,578.4628


In [0]:
import uuid
from pyspark.sql import functions as F

In [0]:

df = df.withColumn("sl_no", F.expr("uuid()"))

display(df.limit(10))

age,gender,bmi,children,discount_eligibility,region,expenses,premium,sl_no
19,female,27.9,0,yes,southwest,16884.92,168.8492,5e967757-4418-434a-aedd-a334c0146c1e
18,male,33.8,1,no,southeast,1725.55,17.2555,6f11b4c3-5155-446e-87ea-d95ee6108e0f
28,male,33.0,3,no,southeast,4449.46,44.4946,7cd2db87-a927-482a-819d-7953afcbe099
33,male,22.7,0,no,northwest,21984.47,439.6894,ba72e1b3-23cd-49c8-961b-ba2a2f7b158f
32,male,28.9,0,no,northwest,3866.86,77.3372,45e4092c-1dd3-4654-b4c4-1c6cc447b929
31,female,25.7,0,no,southeast,3756.62,75.1324,29093839-2c07-44ec-bcae-dc0967485c4c
46,female,33.4,1,no,southeast,8240.59,164.8118,4f1af2ff-bd9b-4f98-8f95-7b7840e7306e
37,female,27.7,3,no,northwest,7281.51,145.6302,2332dbee-6eba-41f0-adbc-bf0067b09a23
37,male,29.8,2,no,northeast,6406.41,128.1282,eb39ec3d-6e19-43b3-afaa-4dadf82d11cd
60,female,25.8,0,no,northwest,28923.14,578.4628,9fb2d66f-171c-4585-a518-111cbb10e5c4


In [0]:
# Reordering columns

first_col = ["sl_no"]

other_col = [i for i in df.columns if i!= "sl_no"]

col_reorder = first_col + other_col

df_reordered = df.select(col_reorder)

display(df_reordered.limit(10))

sl_no,age,gender,bmi,children,discount_eligibility,region,expenses,premium
059a5b85-1109-4b67-9029-a04f4026ee41,19,female,27.9,0,yes,southwest,16884.92,168.8492
c9dc2ff1-210d-4d16-b05a-a8e6877fd568,18,male,33.8,1,no,southeast,1725.55,17.2555
43aef5d6-bfb3-4484-bb7f-c036c5a2690f,28,male,33.0,3,no,southeast,4449.46,44.4946
393cba17-015d-40f2-a4fd-67fa69a3daf1,33,male,22.7,0,no,northwest,21984.47,439.6894
c91ebb39-ebc7-44e5-8b0c-758f3f8405aa,32,male,28.9,0,no,northwest,3866.86,77.3372
dc06f389-66a6-491f-b2ec-cb381a51e0d5,31,female,25.7,0,no,southeast,3756.62,75.1324
8335510a-70b2-40b9-9090-0adf470edafb,46,female,33.4,1,no,southeast,8240.59,164.8118
5ebe2988-5382-4663-82cc-c85f4a747f57,37,female,27.7,3,no,northwest,7281.51,145.6302
5c1531d5-bc88-4b48-8207-871a37c58093,37,male,29.8,2,no,northeast,6406.41,128.1282
d3b04ea9-3511-44ff-a667-ea57a5374b87,60,female,25.8,0,no,northwest,28923.14,578.4628


In [0]:
df_reordered.createOrReplaceTempView("df_reordered")

In [0]:
%%sql
-- To check if sl_no is unique using sql
select count(*) as total_count, count(distinct sl_no) as unique_count
from df_reordered

total_count,unique_count
1338,1338


DataFrame[total_count: bigint, unique_count: bigint]

In [0]:
%%sql
-- Just to show t can be also done using spark sql window
-- function :)

with duplicate as (
    select *,
    row_number() over (
        partition by sl_no order by sl_no desc 
    ) as instances
    from df_reordered
)

select * from duplicate where instances>1

sl_no,age,gender,bmi,children,discount_eligibility,region,expenses,premium,instances


DataFrame[sl_no: string, age: int, gender: string, bmi: double, children: int, discount_eligibility: string, region: string, expenses: double, premium: double, instances: int]

In [0]:
# Checking schema
df_reordered.printSchema()

root
 |-- sl_no: string (nullable = false)
 |-- age: integer (nullable = true)
 |-- gender: string (nullable = true)
 |-- bmi: double (nullable = true)
 |-- children: integer (nullable = true)
 |-- discount_eligibility: string (nullable = true)
 |-- region: string (nullable = true)
 |-- expenses: double (nullable = true)
 |-- premium: double (nullable = true)



In [0]:
# Change discount_eligibility to boolean

df_reordered = (
    df_reordered.withColumn("eligible_for_discount",
        F.expr("try_cast(discount_eligibility as boolean)")
    )
)

df_reordered.printSchema()


root
 |-- sl_no: string (nullable = false)
 |-- age: integer (nullable = true)
 |-- gender: string (nullable = true)
 |-- bmi: double (nullable = true)
 |-- children: integer (nullable = true)
 |-- discount_eligibility: string (nullable = true)
 |-- region: string (nullable = true)
 |-- expenses: double (nullable = true)
 |-- premium: double (nullable = true)
 |-- eligible_for_discount: boolean (nullable = true)



In [0]:
# Checking base stats

display(df_reordered.describe())

summary,sl_no,age,gender,bmi,children,discount_eligibility,region,expenses,premium
count,1338,1338,1338,1338,1338,1338,1338,1338,1338
mean,null,39.20702541106129,null,30.665470852017993,1.0949177877429,null,null,13270.422414050803,262.8746854260093
stddev,null,14.049960379216172,null,6.098382190003366,1.2054927397819095,null,null,12110.011239706457,292.53217841796913
min,00320b34-d3e6-4499-a810-17ec535a0d00,18,female,16.0,0,no,northeast,1121.87,11.2187
max,ffdca9f9-d6e1-4376-802c-f45f914d4d4d,64,male,53.1,5,yes,southwest,63770.43,1983.1064


In [0]:
# check for nulls in all coloumn except sl_no
def null_logic(col_name):
    logic = F.count(
        F.when(
            F.col(col_name).isNull(), col_name)
            ).alias(col_name)
    return(logic)

null_count = df_reordered.select([null_logic(i) for i in col_reorder])

display(null_count)


sl_no,age,gender,bmi,children,discount_eligibility,region,expenses,premium
0,0,0,0,0,0,0,0,0


In [0]:
# Cast premium col to 2 decimal places

df_reordered = (
    df_reordered.withColumn("premium",
    F.round("premium", 2))
)

display(df_reordered.limit(10))

sl_no,age,gender,bmi,children,discount_eligibility,region,expenses,premium,eligible_for_discount
059a5b85-1109-4b67-9029-a04f4026ee41,19,female,27.9,0,yes,southwest,16884.92,168.85,true
c9dc2ff1-210d-4d16-b05a-a8e6877fd568,18,male,33.8,1,no,southeast,1725.55,17.26,false
43aef5d6-bfb3-4484-bb7f-c036c5a2690f,28,male,33.0,3,no,southeast,4449.46,44.49,false
393cba17-015d-40f2-a4fd-67fa69a3daf1,33,male,22.7,0,no,northwest,21984.47,439.69,false
c91ebb39-ebc7-44e5-8b0c-758f3f8405aa,32,male,28.9,0,no,northwest,3866.86,77.34,false
dc06f389-66a6-491f-b2ec-cb381a51e0d5,31,female,25.7,0,no,southeast,3756.62,75.13,false
8335510a-70b2-40b9-9090-0adf470edafb,46,female,33.4,1,no,southeast,8240.59,164.81,false
5ebe2988-5382-4663-82cc-c85f4a747f57,37,female,27.7,3,no,northwest,7281.51,145.63,false
5c1531d5-bc88-4b48-8207-871a37c58093,37,male,29.8,2,no,northeast,6406.41,128.13,false
d3b04ea9-3511-44ff-a667-ea57a5374b87,60,female,25.8,0,no,northwest,28923.14,578.46,false


In [0]:
# Including a random Yes or No Smoker column for depth

df_reordered = (
    df_reordered.withColumn(
        "smoker",
        F.when(F.rand(seed=42)<0.2, "yes")
        .otherwise("no")
    )
)

In [0]:
# Feature Engineering 1
# Binning of age groups

df_age_group= (
    df_reordered.withColumn("life_stage",
    F.when(((F.col("age") >= 18) & (F.col("age") <30)), "youth")
    .when(((F.col("age") >= 30) & (F.col("age") <60)), "adult")
    .otherwise("senior")
    )
)

In [0]:
# Feature Engineering 2
# Risk factor
df_risk = (
    df_age_group.withColumn("risk_factor",
    F.when((F.col("life_stage")== "youth") & (F.col("smoker") == "yes"), "Low")
    .when((F.col("life_stage")== "adult") & (F.col("smoker") == "yes"), "Medium")
    .when((F.col("life_stage")== "senior") & (F.col("smoker") == "yes"), "High")
    .otherwise("very_low")
    )
)

In [0]:
# Feature Engineering 3
# Obesity level
df_final = (
    df_risk.withColumn("obesity",
    F.when((F.col("bmi") < 18.5) , "underweight")
    .when((F.col("bmi") >= 18.5) & (F.col("bmi") < 25), "normal")
    .when((F.col("bmi") >= 25) & (F.col("bmi") < 30), "overweight")
    .otherwise("obese")
    )
)

In [0]:
df_final.printSchema()

root
 |-- sl_no: string (nullable = false)
 |-- age: integer (nullable = true)
 |-- gender: string (nullable = true)
 |-- bmi: double (nullable = true)
 |-- children: integer (nullable = true)
 |-- discount_eligibility: string (nullable = true)
 |-- region: string (nullable = true)
 |-- expenses: double (nullable = true)
 |-- premium: double (nullable = true)
 |-- eligible_for_discount: boolean (nullable = true)
 |-- smoker: string (nullable = false)
 |-- life_stage: string (nullable = false)
 |-- risk_factor: string (nullable = false)
 |-- obesity: string (nullable = false)



In [0]:
df_final.write.format("delta").mode("overwrite").saveAsTable("insurance.dbo.insurance_gold")